In [ ]:

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# ===============================
# paths
# ===============================
snp_path = "highrisk_raw_absIG_tables_CHD/SNP_raw_absIG_high_top_85.csv"
gene_path = "highrisk_raw_absIG_tables_CHD/SNP_GENE_raw_absIG_high_top_85.csv"

rank_col = "rank_by_median"
gene_col = "gene"
snp_col = "feature"

# ===============================
# style (publication-ready)
# ===============================
bar_color = "#4C72B0"   # muted blue
text_color = "#333333"

# ===============================
# CHD GWAS gene list
# ===============================
CHD_GENES = [
    "PCSK9","NOS3","LDLR","LPL","LIPA","ANGPTL4","CDK8","APOB","FN1","CCM2",
    "HNF1A","PNPLA3","REST","SCARB1","APOE","ABCG8","SEMA3C","COL4A1","MECOM",
    "SGCD","ACVRL1","RRP1B","SWAP70","IL6R","NME7","LAMB2","ARHGEF26","EDNRA",
    "GUCY1A3","LOX","PHACTR1","TCF21","PLG","TBX20","ARHGAP42","APOC3","LIPC",
    "MYH11","CETP","SREBF1","CXCR4","MC4R","TGFB1","FES"
]

# ======================================================
# 1️⃣ SNP-level plot
# ======================================================
snp_df = pd.read_csv(snp_path)


snp_df = snp_df[snp_df[snp_col].astype(str).str.startswith("rs")].copy()


rank_cutoff_snp = snp_df[rank_col].max() * 0.65


snp_sub = snp_df[
    (snp_df[gene_col].isin(CHD_GENES)) &
    (snp_df[rank_col] <= rank_cutoff_snp)
].copy()


snp_sub = (
    snp_sub
    .sort_values(rank_col)
    .drop_duplicates(gene_col)
)


max_rank_snp = snp_df[rank_col].max()
snp_sub["reverse_rank"] = max_rank_snp - snp_sub[rank_col] + 1


snp_sub = snp_sub.sort_values("reverse_rank")

# label
snp_sub["label"] = snp_sub[snp_col] + " (" + snp_sub[gene_col] + ")"

# ---------- plot ----------
plt.figure(figsize=(8, 0.35 * len(snp_sub)))
plt.barh(
    snp_sub["label"],
    snp_sub["reverse_rank"],
    color=bar_color
)

for i, r in enumerate(snp_sub[rank_col]):
    plt.text(
        snp_sub["reverse_rank"].iloc[i] * 1.01,
        i,
        f"{int(r)}",
        va="center",
        fontsize=9,
        color=text_color
    )

plt.xlabel("Reverse rank (SNP-level)", color=text_color)
plt.title("CHD GWAS genes in SNP IG ranking (CHD)", color=text_color)
plt.grid(False)
plt.tight_layout()
plt.savefig(
    "fig2/CHD_GWAS_genes_SNP_reverse_rank_barplot.pdf",
    dpi=300,
    transparent=True
)
plt.close()

# ======================================================
# 2️⃣ GENE-level plot
# ======================================================
gene_df = pd.read_csv(gene_path)


rank_cutoff_gene = gene_df[rank_col].max() * 0.65

gene_sub = gene_df[
    (gene_df[gene_col].isin(CHD_GENES)) &
    (gene_df[rank_col] <= rank_cutoff_gene)
].copy()


max_rank_gene = gene_df[rank_col].max()
gene_sub["reverse_rank"] = max_rank_gene - gene_sub[rank_col] + 1

gene_sub = gene_sub.sort_values("reverse_rank")

# ---------- plot ----------
plt.figure(figsize=(8, 0.35 * len(gene_sub)))
plt.barh(
    gene_sub[gene_col],
    gene_sub["reverse_rank"],
    color=bar_color
)

for i, r in enumerate(gene_sub[rank_col]):
    plt.text(
        gene_sub["reverse_rank"].iloc[i] * 1.01,
        i,
        f"{int(r)}",
        va="center",
        fontsize=9,
        color=text_color
    )

plt.xlabel("Reverse rank (GENE-level)", color=text_color)
plt.title("CHD GWAS genes in GENE IG ranking (CHD)", color=text_color)
plt.grid(False)
plt.tight_layout()
plt.savefig(
    "figs4_final/CHD_GWAS_genes_GENE_reverse_rank_barplot.pdf",transparent=True,
    dpi=300
)
plt.close()

print("✅ Finished plotting SNP-level and GENE-level reverse-rank barplots.")

In [ ]:
#SNP Boxplot select Genes from 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# =========================
# Config
# =========================
SNP_IG_CSV = "code_02_02_save/snp_20260202_1432/explanations/CHD_snp_TR_best_IG.csv"
PRED_CSV   = "code_02_02_save/snp_20260202_1432/preds/CHD_snp_TR_test_best_pred.csv"
SNP_MAP_CSV = "fig2/SNP/CHD_IG_final.csv"

HIGH_RISK_Q = 0.85
MAX_X = 0.30
KEEP_REVERSE_FRAC = 0.35
HUE_ORDER = ["High risk", "Others"]


BAR_COLOR = "#8C8C8C"

CHD_GENES = [
    "PCSK9","NOS3","LDLR","LPL","LIPA","ANGPTL4","CDK8","APOB","FN1","CCM2",
    "HNF1A","PNPLA3","REST","SCARB1","APOE","ABCG8","SEMA3C","COL4A1","MECOM",
    "SGCD","ACVRL1","RRP1B","SWAP70","IL6R","NME7","LAMB2","ARHGEF26","EDNRA",
    "GUCY1A3","LOX","PHACTR1","TCF21","PLG","TBX20","ARHGAP42","APOC3","LIPC",
    "MYH11","CETP","SREBF1","CXCR4","MC4R","TGFB1","FES"
]

# =========================
# Load data
# =========================
ig = pd.read_csv(SNP_IG_CSV)
pred = pd.read_csv(PRED_CSV)
pred["pid"] = pred["ID"]

snp_map = pd.read_csv(SNP_MAP_CSV)[["SNP", "gene"]].drop_duplicates()

ig = ig.merge(snp_map, left_on="feature", right_on="SNP", how="inner")
ig["abs_ig"] = ig["ig_abs"].abs()

risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
pred["risk_group"] = np.where(pred["risk"] >= risk_cut, "High risk", "Others")

ig = ig.merge(pred[["pid", "risk_group"]], on="pid", how="inner")

# ======================================================
# =================  SNP LEVEL  =========================
# ======================================================
snp_rank = (
    ig[ig["risk_group"] == "High risk"]
    .groupby(["feature", "gene"], as_index=False)["abs_ig"]
    .median()
    .sort_values("abs_ig", ascending=False)
)

snp_rank["rank_by_median"] = np.arange(1, len(snp_rank) + 1)
snp_rank = snp_rank[snp_rank["gene"].isin(CHD_GENES)]

best_snp = (
    snp_rank
    .sort_values("rank_by_median")
    .drop_duplicates("gene")
)

max_rank = best_snp["rank_by_median"].max()
best_snp["reverse_rank"] = max_rank - best_snp["rank_by_median"] + 1
best_snp["reverse_frac"] = best_snp["reverse_rank"] / max_rank

best_snp = best_snp[
    (best_snp["reverse_frac"] >= KEEP_REVERSE_FRAC) &
    (best_snp["feature"].astype(str).str.startswith("rs"))
]

label_df = (
    best_snp
    .sort_values("reverse_rank", ascending=False)
    .assign(label=lambda x: x["feature"] + " (" + x["gene"] + ")")
)

snp_plot = ig.merge(
    best_snp[["feature", "gene", "rank_by_median", "reverse_rank"]],
    on=["feature", "gene"],
    how="inner"
)
snp_plot["label"] = snp_plot["feature"] + " (" + snp_plot["gene"] + ")"

# ---- SNP plot ----
fig = plt.figure(figsize=(9, 0.5 * len(label_df)))
gs = gridspec.GridSpec(1, 2, width_ratios=[4, 1], wspace=0.05)

ax_box = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1], sharey=ax_box)

sns.boxplot(
    data=snp_plot,
    x="abs_ig",
    y="label",
    hue="risk_group",
    hue_order=HUE_ORDER,
    order=label_df["label"],
    orient="h",
    showfliers=False,
    palette={"High risk": "#C44E52", "Others": "#4C72B0"},
    linewidth=1,
    ax=ax_box
)

sns.stripplot(
    data=snp_plot,
    x="abs_ig",
    y="label",
    hue="risk_group",
    hue_order=HUE_ORDER,
    order=label_df["label"],
    orient="h",
    dodge=True,
    jitter=0.25,
    size=3,
    alpha=0.6,
    edgecolor="black",
    linewidth=0.3,
    palette={"High risk": "#C44E52", "Others": "#4C72B0"},
    ax=ax_box
)

ax_box.set_xlim(0, MAX_X)
ax_box.set_xlabel("IG per sample")
ax_box.set_ylabel("")
ax_box.legend_.remove()


ax_bar.barh(
    label_df["label"],
    label_df["reverse_rank"],
    color=BAR_COLOR
)

for i, r in enumerate(label_df["rank_by_median"]):
    ax_bar.text(
        label_df["reverse_rank"].iloc[i] + 0.5,
        i,
        f"{int(r)}",
        va="center",
        fontsize=9
    )

ax_bar.set_xlabel("Reverse rank")
ax_bar.tick_params(axis="y", left=False, labelleft=False)
ax_bar.grid(False)

for spine in ax_bar.spines.values():
    spine.set_visible(False)

plt.suptitle("CHD GWAS genes – SNP-level |IG| and ranking", y=0.98)
plt.tight_layout()
plt.savefig("figs4_final/fig2B_CHD_GWAS_snp.pdf",transparent=True)

# ======================================================
# =================  GENE LEVEL  ========================
# ======================================================
gene_sample = (
    ig
    .groupby(["pid", "gene", "risk_group"], as_index=False)["abs_ig"]
    .max()
)

gene_rank = (
    gene_sample[gene_sample["risk_group"] == "High risk"]
    .groupby("gene", as_index=False)["abs_ig"]
    .median()
    .sort_values("abs_ig", ascending=False)
)

gene_rank["rank_by_median"] = np.arange(1, len(gene_rank) + 1)
gene_rank = gene_rank[gene_rank["gene"].isin(CHD_GENES)]

max_rank_g = gene_rank["rank_by_median"].max()
gene_rank["reverse_rank"] = max_rank_g - gene_rank["rank_by_median"] + 1
gene_rank["reverse_frac"] = gene_rank["reverse_rank"] / max_rank_g
gene_rank = gene_rank[gene_rank["reverse_frac"] >= KEEP_REVERSE_FRAC]

gene_plot = gene_sample.merge(
    gene_rank[["gene", "rank_by_median", "reverse_rank"]],
    on="gene",
    how="inner"
)

gene_order = (
    gene_rank
    .sort_values("reverse_rank", ascending=False)["gene"]
)

# ---- gene plot ----
fig = plt.figure(figsize=(9, 0.5 * len(gene_order)))
gs = gridspec.GridSpec(1, 2, width_ratios=[4, 1], wspace=0.05)

ax_box = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1], sharey=ax_box)

sns.boxplot(
    data=gene_plot,
    x="abs_ig",
    y="gene",
    hue="risk_group",
    hue_order=HUE_ORDER,
    order=gene_order,
    orient="h",
    showfliers=False,
    palette={"High risk": "#C44E52", "Others": "#4C72B0"},
    linewidth=1,
    ax=ax_box
)

sns.stripplot(
    data=gene_plot,
    x="abs_ig",
    y="gene",
    hue="risk_group",
    hue_order=HUE_ORDER,
    order=gene_order,
    orient="h",
    dodge=True,
    jitter=0.25,
    size=3,
    alpha=0.6,
    edgecolor="black",
    linewidth=0.3,
    palette={"High risk": "#C44E52", "Others": "#4C72B0"},
    ax=ax_box
)

ax_box.set_xlim(0, MAX_X)
ax_box.set_xlabel("IG per sample")
ax_box.set_ylabel("")
ax_box.legend_.remove()

ax_bar.barh(
    gene_order,
    gene_rank.set_index("gene").loc[gene_order]["reverse_rank"],
    color=BAR_COLOR
)

for i, g in enumerate(gene_order):
    r = gene_rank.loc[gene_rank["gene"] == g, "rank_by_median"].values[0]
    ax_bar.text(
        gene_rank.loc[gene_rank["gene"] == g, "reverse_rank"].values[0] + 0.5,
        i,
        f"{int(r)}",
        va="center",
        fontsize=9
    )

ax_bar.set_xlabel("Reverse rank")
ax_bar.tick_params(axis="y", left=False, labelleft=False)
ax_bar.grid(False)
for spine in ax_bar.spines.values():
    spine.set_visible(False)

plt.suptitle("CHD GWAS genes – GENE-level |IG| and ranking", y=0.98)
plt.tight_layout()
plt.savefig("figs4_final/fig2C_CHD_GWAS_gene.pdf",transparent=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
#SNP Boxplot select Genes from 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# ======================================================
# Global config
# ======================================================
HIGH_RISK_Q = 0.5
MAX_X = 0.15
KEEP_REVERSE_FRAC = 0.35
HUE_ORDER = ["High risk", "Others"]

BOX_COLORS = {"High risk": "#C44E52", "Others": "#4C72B0"}
BAR_COLOR = "#5A6F8E"

ROW_HEIGHT = 0.42
FIG_WIDTH = 10

SNP_MAP_CSV = "fig2/SNP/CHD_IG_final.csv"

# ======================================================
# Fixed panel counts (you specified)
# ======================================================
N_CHD_SNP = 14
N_IST_SNP = 6
N_CHD_GENE = 15
N_IST_GENE = 8

# ======================================================
# Gene lists
# ======================================================
CHD_GENES = [
    "PCSK9","NOS3","LDLR","LPL","LIPA","ANGPTL4","CDK8","APOB","FN1","CCM2",
    "HNF1A","PNPLA3","REST","SCARB1","APOE","ABCG8","SEMA3C","COL4A1","MECOM",
    "SGCD","ACVRL1","RRP1B","SWAP70","IL6R","NME7","LAMB2","ARHGEF26","EDNRA",
    "GUCY1A3","LOX","PHACTR1","TCF21","PLG","TBX20","ARHGAP42","APOC3","LIPC",
    "MYH11","CETP","SREBF1","CXCR4","MC4R","TGFB1","FES"
]

ISTROKE_GENES = [
    "CASZ1","PMF1","SEMA4A","PITX2","FGA","FOXF2","HDAC9","TWIST1","CDK6",
    "LINC01492","ABO","CDKN2B-AS1","MMP12","PDE3A","SH2B3","LRCH1","TBX3",
    "ZCCHC14","PRPF8","FURIN","FES","ILF3","SLC44A2"
]

# ======================================================
# Load + preprocess
# ======================================================
def load_ig_and_pred(ig_csv, pred_csv):
    ig = pd.read_csv(ig_csv)
    pred = pd.read_csv(pred_csv)
    pred["pid"] = pred["ID"]

    snp_map = pd.read_csv(SNP_MAP_CSV)[["SNP", "gene"]].drop_duplicates()
    ig = ig.merge(snp_map, left_on="feature", right_on="SNP", how="inner")
    ig["abs_ig"] = ig["ig_abs"].abs()

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
    pred["risk_group"] = np.where(pred["risk"] >= risk_cut, "High risk", "Others")
    ig = ig.merge(pred[["pid", "risk_group"]], on="pid", how="inner")
    return ig

# ======================================================
# Ranking helpers
# ======================================================
def get_best_snps(ig, gene_list):
    snp_rank = (
        ig[ig["risk_group"] == "High risk"]
        .groupby(["feature", "gene"], as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )
    snp_rank["rank_by_median"] = np.arange(1, len(snp_rank) + 1)
    snp_rank = snp_rank[snp_rank["gene"].isin(gene_list)]

    best = snp_rank.sort_values("rank_by_median").drop_duplicates("gene")
    max_rank = best["rank_by_median"].max()
    best["reverse_rank"] = max_rank - best["rank_by_median"] + 1
    best["reverse_frac"] = best["reverse_rank"] / max_rank

    best = best[
        (best["reverse_frac"] >= KEEP_REVERSE_FRAC) &
        (best["feature"].astype(str).str.startswith("rs"))
    ]
    best["label"] = best["feature"] + " (" + best["gene"] + ")"
    return best.sort_values("reverse_rank", ascending=False)

def get_gene_rank(ig, gene_list):
    gene_sample = (
        ig.groupby(["pid", "gene", "risk_group"], as_index=False)["abs_ig"]
        .max()
    )
    gene_rank = (
        gene_sample[gene_sample["risk_group"] == "High risk"]
        .groupby("gene", as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig", ascending=False)
    )
    gene_rank["rank_by_median"] = np.arange(1, len(gene_rank) + 1)
    gene_rank = gene_rank[gene_rank["gene"].isin(gene_list)]

    max_rank = gene_rank["rank_by_median"].max()
    gene_rank["reverse_rank"] = max_rank - gene_rank["rank_by_median"] + 1
    gene_rank["reverse_frac"] = gene_rank["reverse_rank"] / max_rank
    gene_rank = gene_rank[gene_rank["reverse_frac"] >= KEEP_REVERSE_FRAC]

    return gene_sample, gene_rank.sort_values("reverse_rank", ascending=False)

# ======================================================
# Plot helper
# ======================================================
def draw_box_and_rank(ax_box, ax_bar, plot_df, order, rank_df, y_col, title):

    sns.boxplot(
        data=plot_df, x="abs_ig", y=y_col,
        hue="risk_group", hue_order=HUE_ORDER,
        order=order, orient="h", showfliers=False,
        palette=BOX_COLORS, linewidth=1, ax=ax_box
    )

    sns.stripplot(
        data=plot_df, x="abs_ig", y=y_col,
        hue="risk_group", hue_order=HUE_ORDER,
        order=order, orient="h", dodge=True,
        jitter=0.25, size=3, alpha=0.6,
        edgecolor="black", linewidth=0.3,
        palette=BOX_COLORS, ax=ax_box
    )

    ax_box.set_xlim(0, MAX_X)
    ax_box.set_xlabel("Integrated gradient attribution")
    ax_box.set_ylabel("")
    ax_box.legend_.remove()
    ax_box.grid(False)
    ax_box.set_title(title, loc="left", fontsize=12, pad=8)

    ax_bar.barh(order, rank_df["reverse_rank"], color=BAR_COLOR)
    for i, r in enumerate(rank_df["rank_by_median"]):
        ax_bar.text(rank_df["reverse_rank"].iloc[i] + 0.5, i, f"{int(r)}",
                    va="center", fontsize=9)

    ax_bar.set_xlabel("Reverse rank")
    ax_bar.tick_params(axis="y", left=False, labelleft=False)
    ax_bar.grid(False)
    ax_bar.invert_yaxis()
    for s in ax_bar.spines.values():
        s.set_visible(False)

# ======================================================
# Load data
# ======================================================
ig_chd = load_ig_and_pred(
    "code_02_02_save/snp_20260202_1432/explanations/CHD_snp_TR_best_IG.csv",
    "code_02_02_save/snp_20260202_1432/preds/CHD_snp_TR_test_best_pred.csv"
)
ig_istroke = load_ig_and_pred(
    "code_02_02_save/snp_20260202_1432/explanations/istroke_snp_TR_best_IG.csv",
    "code_02_02_save/snp_20260202_1432/preds/istroke_snp_TR_test_best_pred.csv"
)

# ======================================================
# SNP-level figure (height proportional to counts)
# ======================================================
best_chd = get_best_snps(ig_chd, CHD_GENES).head(N_CHD_SNP)
best_ist = get_best_snps(ig_istroke, ISTROKE_GENES).head(N_IST_SNP)

plot_chd = ig_chd.merge(best_chd[["feature","gene","label"]], on=["feature","gene"])
plot_ist = ig_istroke.merge(best_ist[["feature","gene","label"]], on=["feature","gene"])

fig = plt.figure(figsize=(FIG_WIDTH, ROW_HEIGHT * (N_CHD_SNP + N_IST_SNP)))
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[4, 1],
    height_ratios=[N_CHD_SNP, N_IST_SNP],   # ✅ key change
    hspace=0.25, wspace=0.05
)

draw_box_and_rank(fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]),
                  plot_chd, best_chd["label"], best_chd, "label",
                  "Coronary Heart Disease")

draw_box_and_rank(fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]),
                  plot_ist, best_ist["label"], best_ist, "label",
                  "Ischemic Stroke")

plt.savefig("figs4_final/fig2_SNP_CHD_iStroke.pdf",
            dpi=300, bbox_inches="tight", transparent=True)
plt.close()

# ======================================================
# Gene-level figure (height proportional to counts)
# ======================================================
gsample_chd, grank_chd = get_gene_rank(ig_chd, CHD_GENES)
gsample_ist, grank_ist = get_gene_rank(ig_istroke, ISTROKE_GENES)

grank_chd = grank_chd.head(N_CHD_GENE)
grank_ist = grank_ist.head(N_IST_GENE)

plot_g_chd = gsample_chd.merge(grank_chd[["gene"]], on="gene")
plot_g_ist = gsample_ist.merge(grank_ist[["gene"]], on="gene")

fig = plt.figure(figsize=(FIG_WIDTH, ROW_HEIGHT * (N_CHD_GENE + N_IST_GENE)))
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[4, 1],
    height_ratios=[N_CHD_GENE, N_IST_GENE],  # ✅ key change
    hspace=0.25, wspace=0.05
)

draw_box_and_rank(fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]),
                  plot_g_chd, grank_chd["gene"], grank_chd, "gene",
                  "Coronary Heart Disease")

draw_box_and_rank(fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1]),
                  plot_g_ist, grank_ist["gene"], grank_ist, "gene",
                  "Ischemic Stroke")

plt.savefig("figs4_final/figs4_final_GENE_CHD_iStroke.pdf",
            dpi=300, bbox_inches="tight", transparent=True)
plt.close()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
#SNP Boxplot select Genes from 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# ======================================================
# Config
# ======================================================

ENDPOINTS = ["CHD", "istroke"]

SNP_IG_TEMPLATE = "code_02_02_save/snp_20260202_1432/explanations/{}_snp_TR_best_IG.csv"
PRED_TEMPLATE   = "code_02_02_save/snp_20260202_1432/preds/{}_snp_TR_test_best_pred.csv"

SNP_MAP = "fig2/SNP/CHD_IG_final.csv"

OUTDIR = "figs4_final"
os.makedirs(OUTDIR, exist_ok=True)

HIGH_RISK_Q = 0.85
TOP_N_GENES = 10
MAX_X = 0.30


BOX_PALETTE = {
    "High risk": "#F3B0B2",
    "Others": "#AFC7E8"
}


POINT_PALETTE = {
    "High risk": "#C44E52",
    "Others": "#4C72B0"
}

# GSEA settings
GENESETS = [
    "MSigDB_Hallmark_2020",
    "GO_Biological_Process_2023",
    "KEGG_2021_Human"
]

PERM = 2000
MIN_SIZE = 10
MAX_SIZE = 1000
SEED = 42

sns.set_style("white")

# ======================================================
# SNP → gene map
# ======================================================

snp_map = pd.read_csv(SNP_MAP)[["SNP","gene"]].drop_duplicates()

# ======================================================
# Main loop
# ======================================================

for ENDPOINT in ENDPOINTS:

    print("====================================")
    print("Running:", ENDPOINT)
    print("====================================")

    SNP_IG_CSV = SNP_IG_TEMPLATE.format(ENDPOINT)
    PRED_CSV   = PRED_TEMPLATE.format(ENDPOINT)

    ig = pd.read_csv(SNP_IG_CSV)
    pred = pd.read_csv(PRED_CSV)

    pred["pid"] = pred["ID"]

    ig = ig.merge(
        snp_map,
        left_on="feature",
        right_on="SNP",
        how="inner"
    )

    # abs IG
    if "ig_abs" in ig.columns:
        ig["abs_ig"] = ig["ig_abs"].abs()
    else:
        ig["abs_ig"] = ig["ig"].abs()

    # ======================================================
    # risk group
    # ======================================================

    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)

    pred["risk_group"] = np.where(
        pred["risk"] >= risk_cut,
        "High risk",
        "Others"
    )

    ig = ig.merge(
        pred[["pid","risk_group"]],
        on="pid",
        how="inner"
    )

    # ======================================================
    # gene-level aggregation
    # ======================================================

    gene_sample = (
        ig
        .groupby(["pid","gene","risk_group"],as_index=False)["abs_ig"]
        .max()
    )

    # ======================================================
    # ranking genes (High risk median IG)
    # ======================================================

    gene_rank = (
        gene_sample[gene_sample["risk_group"]=="High risk"]
        .groupby("gene",as_index=False)["abs_ig"]
        .median()
        .sort_values("abs_ig",ascending=False)
    )

    # ======================================================
    # remove ENSG / Metazoa
    # ======================================================

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("ENSG",na=False)
    ]
    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("LINC",na=False)
    ]
    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("Y_",na=False)
    ]
    gene_rank = gene_rank[
        ~gene_rank["gene"].str.startswith("RNF",na=False)
    ]

    gene_rank = gene_rank[
        ~gene_rank["gene"].str.contains("Metazoa",na=False)
    ]

    # ======================================================
    # Top genes
    # ======================================================

    top_genes = gene_rank["gene"].head(TOP_N_GENES).tolist()

    gene_plot = gene_sample[
        gene_sample["gene"].isin(top_genes)
    ]

    gene_order = (
        gene_rank
        .set_index("gene")
        .loc[top_genes]
        .sort_values("abs_ig",ascending=False)
        .index
        .tolist()
    )

    # ======================================================
    # Plot
    # ======================================================

    plt.figure(figsize=(4,0.4*len(gene_order)+1))

    ax = sns.boxplot(
        data=gene_plot,
        x="abs_ig",
        y="gene",
        hue="risk_group",
        hue_order=["High risk","Others"],
        order=gene_order,
        orient="h",
        showfliers=False,
        palette=BOX_PALETTE,
        linewidth=1
    )

    sns.stripplot(
        data=gene_plot,
        x="abs_ig",
        y="gene",
        hue="risk_group",
        hue_order=["High risk","Others"],
        order=gene_order,
        orient="h",
        dodge=True,
        jitter=0.25,
        size=3,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
        palette=POINT_PALETTE
    )

    # remove duplicated legends
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], frameon=False)

    plt.xlim(0,0.2)

    #plt.xlabel("Integrated gradient attribution")
    plt.ylabel("")

    #plt.title(f"{ENDPOINT} – Top {TOP_N_GENES} genes (median |IG| in high risk)")

    plt.tight_layout()

    plt.savefig(
        f"{OUTDIR}/{ENDPOINT}_top_gene_origin_boxplot.pdf",
        transparent=True
    )

    plt.close()

In [ ]:

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

# =========================
# Config
# =========================
#TARGET_SNP = "rs1327115596"
TARGET_SNP = "rs662145"
WINDOW_BP = 1_000_000_0

IG_CSV = "code_02_02_save/snp_20260202_1432/explanations/CHD_snp_TR_best_IG.csv"
PRED_CSV = "code_02_02_save/snp_20260202_1432/preds/CHD_snp_TR_test_best_pred.csv"

# NEW: variant → position table
VARIANT_CSV = "CHD_SNP_variant_gene_with_symbol.csv"

USE_HIGH_RISK_ONLY = True
HIGH_RISK_Q = 0.85

OUTFIG = "figs4_final/PCSK9_rs662145_locus_IG.pdf"

# =========================
# Load data
# =========================
ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)
pred["pid"] = pred["ID"]

variant = pd.read_csv(VARIANT_CSV)

# =========================
# Parse chr / pos from variant_id
# variant_id example: chr1_1330959_C_T_b38
# =========================
def parse_variant_id(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    else:
        return np.nan, np.nan

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant_id(x))
)

# sanity check
variant = variant.dropna(subset="variant_id").reset_index()
assert variant["chr"].notna().all(), "Some variant_id failed to parse chr"
assert variant["pos"].notna().all(), "Some variant_id failed to parse pos"

# =========================
# Merge IG with variant position
# =========================
ig = ig.merge(
    variant[["SNP", "chr", "pos", "coding_gene"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)


ig["abs_ig"] = ig["ig_abs"].abs()

# =========================
# High-risk filtering
# =========================
if USE_HIGH_RISK_ONLY:
    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
    high_pid = pred.loc[pred["risk"] >= risk_cut, "pid"]
    ig = ig[ig["pid"].isin(high_pid)].copy()

# =========================
# Locate target SNP
# =========================
target_row = variant[variant["SNP"] == TARGET_SNP]
assert len(target_row) == 1, "Target SNP not uniquely found"

target_chr = target_row["chr"].iloc[0]
target_pos = target_row["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# =========================
# Subset locus
# =========================
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

print(f"Number of SNPs in locus: {locus['SNP'].nunique()}")

# =========================
# Aggregate IG per SNP
# =========================
snp_agg = (
    locus
    .groupby(["SNP", "chr", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

# =========================
# Define region type
# =========================
def infer_region(coding_gene):
    if pd.isna(coding_gene) or coding_gene == "":
        return "non-coding"
    else:
        return "coding"

df_region = pd.read_csv("fig2/SNP/CHD_IG_final.csv")
region_dict = {}
for i in range(len(df_region)):
    region_dict[df_region["SNP"][i]]= df_region["region"][i]

#snp_agg["region"] = snp_agg["coding_gene"].apply(infer_region)
snp_agg["region"] = snp_agg["SNP"]
for i in range(len(snp_agg)):
    snp_agg["region"][i] = ""
    snp_agg["region"][i] = region_dict[snp_agg["SNP"][i]]
# =========================
# Plot
# =========================
plt.figure(figsize=(10, 4))

palette = {
    "coding": "#C44E52",
    "intergenic": "#4C72B0",
    "promoter": "black"
}

sns.scatterplot(
    data=snp_agg,
    x="pos",
    y="mean_abs_ig",
    hue="region",
    palette=palette,
    edgecolor="black",
    linewidth=0.3,
    s=35
)

# highlight target SNP
plt.axvline(
    target_pos,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.text(
    target_pos,
    snp_agg["mean_abs_ig"].max() * 0.95,
    TARGET_SNP,
    ha="center",
    va="top",
    fontsize=9,
    rotation=90
)

plt.xlabel(f"{target_chr} position (bp)")
plt.ylabel("Mean |Integrated Gradient|")
plt.title(f"PCSK9 locus ±10 Mb around {TARGET_SNP} (Coronary Heart Disease)")

#plt.legend(title="Region", frameon=False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True)
plt.show()

In [ ]:

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# =========================
# Config
# =========================
#TARGET_SNP = "rs1327115596"
TARGET_SNP = "rs3782805"
WINDOW_BP = 1_000_000_0

IG_CSV = "code_02_02_save/snp_20260202_1432/explanations/istroke_snp_TR_best_IG.csv"
PRED_CSV = "code_02_02_save/snp_20260202_1432/preds/istroke_snp_TR_test_best_pred.csv"

# NEW: variant → position table
VARIANT_CSV = "CHD_SNP_variant_gene_with_symbol.csv"

USE_HIGH_RISK_ONLY = True
HIGH_RISK_Q = 0.85

OUTFIG = "figs4_final/TSPAN9_rs3782805_locus_IG.pdf"

# =========================
# Load data
# =========================
ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)
pred["pid"] = pred["ID"]

variant = pd.read_csv(VARIANT_CSV)

# =========================
# Parse chr / pos from variant_id
# variant_id example: chr1_1330959_C_T_b38
# =========================
def parse_variant_id(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    else:
        return np.nan, np.nan

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant_id(x))
)

# sanity check
variant = variant.dropna(subset="variant_id").reset_index()
assert variant["chr"].notna().all(), "Some variant_id failed to parse chr"
assert variant["pos"].notna().all(), "Some variant_id failed to parse pos"

# =========================
# Merge IG with variant position
# =========================
ig = ig.merge(
    variant[["SNP", "chr", "pos", "coding_gene"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)


ig["abs_ig"] = ig["ig_abs"].abs()

# =========================
# High-risk filtering
# =========================
if USE_HIGH_RISK_ONLY:
    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
    high_pid = pred.loc[pred["risk"] >= risk_cut, "pid"]
    ig = ig[ig["pid"].isin(high_pid)].copy()

# =========================
# Locate target SNP
# =========================
target_row = variant[variant["SNP"] == TARGET_SNP]
assert len(target_row) == 1, "Target SNP not uniquely found"

target_chr = target_row["chr"].iloc[0]
target_pos = target_row["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# =========================
# Subset locus
# =========================
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

print(f"Number of SNPs in locus: {locus['SNP'].nunique()}")

# =========================
# Aggregate IG per SNP
# =========================
snp_agg = (
    locus
    .groupby(["SNP", "chr", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

# =========================
# Define region type
# =========================
def infer_region(coding_gene):
    if pd.isna(coding_gene) or coding_gene == "":
        return "non-coding"
    else:
        return "coding"

df_region = pd.read_csv("fig2/SNP/CHD_IG_final.csv")
region_dict = {}
for i in range(len(df_region)):
    region_dict[df_region["SNP"][i]]= df_region["region"][i]

#snp_agg["region"] = snp_agg["coding_gene"].apply(infer_region)
snp_agg["region"] = snp_agg["SNP"]
for i in range(len(snp_agg)):
    snp_agg["region"][i] = ""
    snp_agg["region"][i] = region_dict[snp_agg["SNP"][i]]
# =========================
# Plot
# =========================
plt.figure(figsize=(10, 4))

palette = {
    "coding": "#C44E52",
    "intergenic": "#4C72B0",
    "promoter": "black"
}

sns.scatterplot(
    data=snp_agg,
    x="pos",
    y="mean_abs_ig",
    hue="region",
    palette=palette,
    edgecolor="black",
    linewidth=0.3,
    s=35
)

# highlight target SNP
plt.axvline(
    target_pos,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.text(
    target_pos,
    snp_agg["mean_abs_ig"].max() * 0.95,
    TARGET_SNP,
    ha="center",
    va="top",
    fontsize=9,
    rotation=90
)

plt.xlabel(f"{target_chr} position (bp)")
plt.ylabel("Mean |Integrated Gradient|")
plt.title(f"TSPAN9 locus ±10 Mb around {TARGET_SNP} (Ischemic Stroke)")

#plt.legend(title="Region", frameon=False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True)
plt.show()

In [ ]:

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# =========================
# Config
# =========================
TARGET_SNP = "rs1327115596"
#TARGET_SNP = "rs662145"
WINDOW_BP = 1_000_000_0

IG_CSV = "code_02_02_save/snp_20260202_1432/explanations/CHD_snp_TR_best_IG.csv"
PRED_CSV = "code_02_02_save/snp_20260202_1432/preds/CHD_snp_TR_test_best_pred.csv"

# NEW: variant → position table
VARIANT_CSV = "CHD_SNP_variant_gene_with_symbol.csv"

USE_HIGH_RISK_ONLY = True
HIGH_RISK_Q = 0.85

OUTFIG = "figs4_final/APOC3_rs1327115596_locus_IG.pdf"

# =========================
# Load data
# =========================
ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)
pred["pid"] = pred["ID"]

variant = pd.read_csv(VARIANT_CSV)

# =========================
# Parse chr / pos from variant_id
# variant_id example: chr1_1330959_C_T_b38
# =========================
def parse_variant_id(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    else:
        return np.nan, np.nan

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant_id(x))
)

# sanity check
variant = variant.dropna(subset="variant_id").reset_index()
assert variant["chr"].notna().all(), "Some variant_id failed to parse chr"
assert variant["pos"].notna().all(), "Some variant_id failed to parse pos"

# =========================
# Merge IG with variant position
# =========================
ig = ig.merge(
    variant[["SNP", "chr", "pos", "coding_gene"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)


ig["abs_ig"] = ig["ig_abs"].abs()

# =========================
# High-risk filtering
# =========================
if USE_HIGH_RISK_ONLY:
    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
    high_pid = pred.loc[pred["risk"] >= risk_cut, "pid"]
    ig = ig[ig["pid"].isin(high_pid)].copy()

# =========================
# Locate target SNP
# =========================
target_row = variant[variant["SNP"] == TARGET_SNP]
assert len(target_row) == 1, "Target SNP not uniquely found"

target_chr = target_row["chr"].iloc[0]
target_pos = target_row["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# =========================
# Subset locus
# =========================
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

print(f"Number of SNPs in locus: {locus['SNP'].nunique()}")

# =========================
# Aggregate IG per SNP
# =========================
snp_agg = (
    locus
    .groupby(["SNP", "chr", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

# =========================
# Define region type
# =========================
def infer_region(coding_gene):
    if pd.isna(coding_gene) or coding_gene == "":
        return "non-coding"
    else:
        return "coding"

df_region = pd.read_csv("fig2/SNP/CHD_IG_final.csv")
region_dict = {}
for i in range(len(df_region)):
    region_dict[df_region["SNP"][i]]= df_region["region"][i]

#snp_agg["region"] = snp_agg["coding_gene"].apply(infer_region)
snp_agg["region"] = snp_agg["SNP"]
for i in range(len(snp_agg)):
    snp_agg["region"][i] = ""
    snp_agg["region"][i] = region_dict[snp_agg["SNP"][i]]
# =========================
# Plot
# =========================
plt.figure(figsize=(10, 4))

palette = {
    "coding": "#C44E52",
    "intergenic": "#4C72B0",
    "promoter": "black"
}

sns.scatterplot(
    data=snp_agg,
    x="pos",
    y="mean_abs_ig",
    hue="region",
    palette=palette,
    edgecolor="black",
    linewidth=0.3,
    s=35
)

# highlight target SNP
plt.axvline(
    target_pos,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.text(
    target_pos,
    snp_agg["mean_abs_ig"].max() * 0.95,
    TARGET_SNP,
    ha="center",
    va="top",
    fontsize=9,
    rotation=90
)

plt.xlabel(f"{target_chr} position (bp)")
plt.ylabel("Mean |Integrated Gradient|")
plt.title(f"APOC3 locus ±1 Mb around {TARGET_SNP} (Coronary Heart Disease)")

plt.legend(title="Region", frameon=False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True)
plt.show()

In [ ]:

import os
import re
import warnings
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ======================================================
# Global settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

sns.set_style("white")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
# =========================
# Config
# =========================
#TARGET_SNP = "rs1327115596"
TARGET_SNP = "rs2089990"
WINDOW_BP = 1_000_000_0

IG_CSV = "code_02_02_save/snp_20260202_1432/explanations/istroke_snp_TR_best_IG.csv"
PRED_CSV = "code_02_02_save/snp_20260202_1432/preds/istroke_snp_TR_test_best_pred.csv"

# NEW: variant → position table
VARIANT_CSV = "CHD_SNP_variant_gene_with_symbol.csv"

USE_HIGH_RISK_ONLY = True
HIGH_RISK_Q = 0.85

OUTFIG = "figs4_final/GRID2_rs2089990_locus_IG.pdf"

# =========================
# Load data
# =========================
ig = pd.read_csv(IG_CSV)
pred = pd.read_csv(PRED_CSV)
pred["pid"] = pred["ID"]

variant = pd.read_csv(VARIANT_CSV)

# =========================
# Parse chr / pos from variant_id
# variant_id example: chr1_1330959_C_T_b38
# =========================
def parse_variant_id(v):
    m = re.match(r"(chr[0-9XY]+)_(\d+)_", str(v))
    if m:
        return m.group(1), int(m.group(2))
    else:
        return np.nan, np.nan

variant[["chr", "pos"]] = variant["variant_id"].apply(
    lambda x: pd.Series(parse_variant_id(x))
)

# sanity check
variant = variant.dropna(subset="variant_id").reset_index()
assert variant["chr"].notna().all(), "Some variant_id failed to parse chr"
assert variant["pos"].notna().all(), "Some variant_id failed to parse pos"

# =========================
# Merge IG with variant position
# =========================
ig = ig.merge(
    variant[["SNP", "chr", "pos", "coding_gene"]],
    left_on="feature",
    right_on="SNP",
    how="inner"
)


ig["abs_ig"] = ig["ig_abs"].abs()

# =========================
# High-risk filtering
# =========================
if USE_HIGH_RISK_ONLY:
    risk_cut = pred["risk"].quantile(HIGH_RISK_Q)
    high_pid = pred.loc[pred["risk"] >= risk_cut, "pid"]
    ig = ig[ig["pid"].isin(high_pid)].copy()

# =========================
# Locate target SNP
# =========================
target_row = variant[variant["SNP"] == TARGET_SNP]
assert len(target_row) == 1, "Target SNP not uniquely found"

target_chr = target_row["chr"].iloc[0]
target_pos = target_row["pos"].iloc[0]

left = target_pos - WINDOW_BP
right = target_pos + WINDOW_BP

# =========================
# Subset locus
# =========================
locus = ig[
    (ig["chr"] == target_chr) &
    (ig["pos"] >= left) &
    (ig["pos"] <= right)
].copy()

print(f"Number of SNPs in locus: {locus['SNP'].nunique()}")

# =========================
# Aggregate IG per SNP
# =========================
snp_agg = (
    locus
    .groupby(["SNP", "chr", "pos"], as_index=False)["abs_ig"]
    .mean()
    .rename(columns={"abs_ig": "mean_abs_ig"})
)

# =========================
# Define region type
# =========================
def infer_region(coding_gene):
    if pd.isna(coding_gene) or coding_gene == "":
        return "non-coding"
    else:
        return "coding"

df_region = pd.read_csv("fig2/SNP/CHD_IG_final.csv")
region_dict = {}
for i in range(len(df_region)):
    region_dict[df_region["SNP"][i]]= df_region["region"][i]

#snp_agg["region"] = snp_agg["coding_gene"].apply(infer_region)
snp_agg["region"] = snp_agg["SNP"]
for i in range(len(snp_agg)):
    snp_agg["region"][i] = ""
    snp_agg["region"][i] = region_dict[snp_agg["SNP"][i]]
# =========================
# Plot
# =========================
plt.figure(figsize=(10, 4))

palette = {
    "coding": "#C44E52",
    "intergenic": "#4C72B0",
    "promoter": "black"
}

sns.scatterplot(
    data=snp_agg,
    x="pos",
    y="mean_abs_ig",
    hue="region",
    palette=palette,
    edgecolor="black",
    linewidth=0.3,
    s=35
)

# highlight target SNP
plt.axvline(
    target_pos,
    color="black",
    linestyle="--",
    linewidth=1
)

plt.text(
    target_pos,
    snp_agg["mean_abs_ig"].max() * 0.95,
    TARGET_SNP,
    ha="center",
    va="top",
    fontsize=9,
    rotation=90
)

plt.xlabel(f"{target_chr} position (bp)")
plt.ylabel("Mean |Integrated Gradient|")
plt.title(f"GRID2 locus ±10 Mb around {TARGET_SNP} (Ischemic Stroke)")

#plt.legend(title="Region", frameon=False)
plt.tight_layout()
plt.savefig(OUTFIG, dpi=300, transparent=True)
plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import matplotlib.colors as mcolors
import matplotlib as mpl
import textwrap

# ---------------------------------------------------------
# Output directory
# ---------------------------------------------------------

OUTDIR = "figs5_final"
os.makedirs(OUTDIR, exist_ok=True)

# ---------------------------------------------------------
# Global style
# ---------------------------------------------------------

plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["font.family"] = "Times New Roman"

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


def wrap_label(text, width=14):
    return textwrap.fill(text, width=width)


def clean_term(t):
    return t.rsplit("(", 1)[0].strip().title()


# ---------------------------------------------------------
# Selected pathways
# ---------------------------------------------------------

selected_terms = [
    "reverse cholesterol transport (GO:0043691)",
    "high-density lipoprotein particle remodeling (GO:0034375)",
    "triglyceride homeostasis (GO:0070328)",
    "acylglycerol metabolic process (GO:0006639)",
    "atrial cardiac muscle cell membrane repolarization (GO:0099624)",
    "regulation of cation channel activity (GO:2001257)",
    "axon guidance (GO:0007411)",
    "regulation of potassium ion transport (GO:0043266)",
    "regulation of heart contraction (GO:0008016)",
    "regulation of cell communication by electrical coupling (GO:0010649)",
    "regulation of cholesterol metabolic process (GO:0090181)",
    "retinal ganglion cell axon guidance (GO:0031290)"
]

# ---------------------------------------------------------
# Load GO files
# ---------------------------------------------------------

files = {
    "CHD": os.path.join(OUTDIR, "top5%_GO_Biological_Process_2021_CHD_SNP.csv"),
    "Stroke": os.path.join(OUTDIR, "top5%_GO_Biological_Process_2021_istroke_SNP.csv")
}

dfs = {}
alpha = 1#0.05

for disease, path in files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(path)

    if "Adjusted P-value" not in df.columns:
        raise ValueError(f"{path} does not contain column: Adjusted P-value")
    if "Term" not in df.columns:
        raise ValueError(f"{path} does not contain column: Term")
    if "Genes" not in df.columns:
        raise ValueError(f"{path} does not contain column: Genes")

    df = df[df["Adjusted P-value"] < alpha].copy()
    df = df[df["Term"].isin(selected_terms)].copy()

    dfs[disease] = df

# ---------------------------------------------------------
# Venn gene sets
# ---------------------------------------------------------

CHD_genes = {
    g.strip()
    for gs in dfs["CHD"]["Genes"].dropna()
    for g in str(gs).split(";")
    if g.strip() != ""
}

Stroke_genes = {
    g.strip()
    for gs in dfs["Stroke"]["Genes"].dropna()
    for g in str(gs).split(";")
    if g.strip() != ""
}

# ---------------------------------------------------------
# Color scheme
# ---------------------------------------------------------

CHD_color = "#C97C5D"
Stroke_color = "#6B8F71"
deep_red = "#B00020"


def mix(colors):
    return np.mean([mcolors.to_rgb(c) for c in colors], axis=0)


region_color_map = {
    "10": CHD_color,
    "01": Stroke_color,
    "11": deep_red,
}

# region rank for bold logic
region_rank = {
    "11": 0,
    "10": 1,
    "01": 1
}


def region_of_gene(g):
    return ("1" if g in CHD_genes else "0") + \
           ("1" if g in Stroke_genes else "0")


gene_region_color = {
    g: region_color_map[region_of_gene(g)]
    for g in (CHD_genes | Stroke_genes)
    if region_of_gene(g) in region_color_map
}

# ---------------------------------------------------------
# Gene lists & count
# ---------------------------------------------------------

gene_lists = {}
gene_counts = {}

for term in selected_terms:
    genes = set()

    for dis, df in dfs.items():
        tmp = df[df["Term"] == term]

        for gs in tmp["Genes"].dropna():
            genes.update([
                x.strip()
                for x in str(gs).split(";")
                if x.strip() != ""
            ])

    gene_lists[term] = sorted(list(genes))
    gene_counts[term] = len(genes)

# ---------------------------------------------------------
# Bubble plot values — logp=None means draw as empty circle
# ---------------------------------------------------------

bubble_values = {}

for disease, df in dfs.items():
    bubble_values[disease] = {}

    for term in selected_terms:
        row = df[df["Term"] == term]

        if len(row) == 0:
            bubble_values[disease][term] = (None, gene_counts[term])
        else:
            p = row["Adjusted P-value"].values[0]
            bubble_values[disease][term] = (-np.log10(p), gene_counts[term])

# ---------------------------------------------------------
# Color normalization for bubble plot
# ---------------------------------------------------------

all_logp = []

for disease in bubble_values:
    for term in selected_terms:
        logp, _ = bubble_values[disease][term]
        if logp is not None and np.isfinite(logp):
            all_logp.append(logp)

if len(all_logp) > 0:
    vmin = min(all_logp)
    vmax = max(all_logp)
else:
    vmin, vmax = 0, 1

norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.viridis

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------

fig = plt.figure(figsize=(12, 12))

# =========================================================
# A. Venn
# =========================================================

venn_ax = plt.subplot2grid((10, 1), (0, 0), rowspan=2)

v = venn2(
    [CHD_genes, Stroke_genes],
    set_labels=("Coronary Heart Disease", "Ischemic Stroke"),
    ax=venn_ax
)

for id_, color in region_color_map.items():
    p = v.get_patch_by_id(id_)
    if p:
        p.set_color(color)
        p.set_alpha(0.55)
        p.set_edgecolor("white")
        p.set_linewidth(1.0)

for text in v.set_labels:
    if text:
        text.set_fontsize(16)
        text.set_fontweight("bold")

for text in v.subset_labels:
    if text:
        text.set_fontsize(14)
        text.set_fontweight("bold")

venn_ax.set_title("Gene Overlap (padj < 0.05)", fontsize=20)

# =========================================================
# B. Barplot
# =========================================================

bar_ax = plt.subplot2grid((10, 1), (3, 0), rowspan=3)

bars = bar_ax.bar(
    range(len(selected_terms)),
    [gene_counts[t] for t in selected_terms],
    color=(0.9, 0.9, 0.9, 0.6)
)

bar_ax.spines["top"].set_visible(False)
bar_ax.spines["right"].set_visible(False)
bar_ax.set_ylabel("Gene Count", fontsize=14, fontweight="bold")
bar_ax.set_xticks([])

# gene labels
for j, term in enumerate(selected_terms):
    genes = gene_lists[term]

    genes_sorted = sorted(
        genes,
        key=lambda g: region_rank.get(region_of_gene(g), 9)
    )

    for k, g in enumerate(genes_sorted):

        if g not in gene_region_color:
            continue

        region = region_of_gene(g)
        bold_flag = region_rank.get(region, 9) <= 1

        bar_ax.text(
            j,
            0.3 + k * 0.9,
            g,
            fontsize=8,
            ha="center",
            color=gene_region_color[g],
            fontweight="bold" if bold_flag else "normal"
        )

# =========================================================
# C. Bubble plot
# =========================================================

mat_ax = plt.subplot2grid((10, 1), (7, 0), rowspan=3)

diseases_order = ["Stroke", "CHD"]
y_pos = [0.04, 0.10]
mat_ax.set_ylim(0.0, 0.14)

for yi, dis in enumerate(diseases_order):
    for xi, term in enumerate(selected_terms):

        logp, gcount = bubble_values[dis][term]

        size = gcount * 90

        if gcount == 0:
            size = 60

        if logp is None:
            # empty circle
            mat_ax.scatter(
                xi,
                y_pos[yi],
                s=size,
                facecolors="none",
                edgecolors="gray",
                linewidth=0.8
            )
        else:
            # filled circle
            mat_ax.scatter(
                xi,
                y_pos[yi],
                s=size,
                c=[logp],
                cmap=cmap,
                norm=norm,
                edgecolors="black",
                linewidth=0.25
            )

mat_ax.set_yticks(y_pos)
mat_ax.set_yticklabels(
    ["Ischemic Stroke", "Coronary Heart Disease"],
    fontsize=13
)

wrapped = [
    wrap_label(clean_term(t), width=14)
    for t in selected_terms
]

mat_ax.set_xticks(range(len(selected_terms)))
mat_ax.set_xticklabels(
    wrapped,
    fontsize=6,
    rotation=45,
    ha="right"
)

mat_ax.spines["top"].set_visible(False)
mat_ax.spines["right"].set_visible(False)

# colorbar
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = plt.colorbar(
    sm,
    ax=mat_ax,
    pad=0.02,
    fraction=0.03
)
cbar.set_label("-log10(padj)", fontsize=12)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

plt.tight_layout()

out_pdf = os.path.join(
    OUTDIR,
    "FINAL_VENN_BAR_BUBBLE_CHD_istroke_v1.pdf"
)

out_svg = os.path.join(
    OUTDIR,
    "FINAL_VENN_BAR_BUBBLE_CHD_istroke_v1.svg"
)

plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches="tight")
plt.savefig(out_svg, dpi=300, transparent=True, bbox_inches="tight")

print(f"Saved: {out_pdf}")
print(f"Saved: {out_svg}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# =========================================================
# Global style
# =========================================================

plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["font.family"] = "Times New Roman"

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"


def wrap_label(text, width=18):
    return textwrap.fill(text, width=width)


def clean_term(t):
    return t.rsplit("(", 1)[0].strip().title()


def term_key(t):
    
    return t.rsplit("(", 1)[0].strip().lower()


# =========================================================

# =========================================================

OUTDIR = "figs5_final"
os.makedirs(OUTDIR, exist_ok=True)


# =========================================================


# =========================================================

SOMA_FILES = {
    "CHD": os.path.join(OUTDIR, "CHD_transformer_h256_l8_IG.csv"),
    "Stroke": os.path.join(OUTDIR, "istroke_transformer_h256_l8_IG.csv")
}

TOP_PCT = 0.10
ALPHA = 0.05


# =========================================================

# =========================================================

gene_sets = {}

for disease, path in SOMA_FILES.items():

    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find Soma IG file: {path}")

    df = pd.read_csv(path)

    if "feature" not in df.columns:
        raise RuntimeError(f"{path} missing column: feature")

    if "importance" not in df.columns:
        raise RuntimeError(f"{path} missing column: importance")

    df["gene"] = (
        df["feature"]
        .astype(str)
        .str.replace("^seq\\.", "", regex=True)
        .str.split(".")
        .str[0]
    )

    df = df.sort_values("importance", ascending=False)

    n_top = max(1, int(len(df) * TOP_PCT))

    genes = (
        df.head(n_top)["gene"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    gene_sets[disease] = set(genes)

    print(f"[OK] {disease}: {len(genes)} top Soma genes")


# =========================================================


# =========================================================

dfs_full = {}
dfs_top = {}

for disease in SOMA_FILES.keys():

    go_path = os.path.join(OUTDIR, f"{disease}_GO_full.csv")

    if not os.path.exists(go_path):
        raise FileNotFoundError(f"Cannot find GO file: {go_path}")

    df = pd.read_csv(go_path)

    if "Adjusted P-value" not in df.columns:
        raise RuntimeError(f"{go_path} missing 'Adjusted P-value' column")

    if "Term" not in df.columns:
        raise RuntimeError(f"{go_path} missing 'Term' column")

    if "Genes" not in df.columns:
        raise RuntimeError(f"{go_path} missing 'Genes' column")

    df = df[df["Adjusted P-value"] < ALPHA].copy()
    df["Disease"] = disease

    dfs_full[disease] = df.copy()

    if "Combined Score" in df.columns:
        df_top = df.sort_values("Combined Score", ascending=False).head(50).copy()
    else:
        df_top = df.sort_values("Adjusted P-value", ascending=True).head(50).copy()

    dfs_top[disease] = df_top

    print(f"[OK] {disease}: {len(df)} significant GO terms (<{ALPHA})")



df_all = pd.concat(dfs_full.values(), ignore_index=True)
merged_path = os.path.join(OUTDIR, "merged_GO_for_plot_CHD_istroke.csv")
df_all.to_csv(merged_path, index=False)
print(f"[OK] {merged_path} saved")


# =========================================================

# =========================================================

def select_terms_auto(dfs_full, k=10):
    rows = []

    for dis, df in dfs_full.items():
        for _, r in df.iterrows():

            combined = r.get("Combined Score", np.nan)

            rows.append({
                "Disease": dis,
                "Term": r["Term"],
                "TermKey": term_key(r["Term"]),
                "padj": float(r["Adjusted P-value"]),
                "Combined": float(combined) if pd.notna(combined) else np.nan
            })

    dd = pd.DataFrame(rows)

    if dd.empty:
        return []

    gp = dd.groupby("TermKey").agg(
        appear=("Disease", "nunique"),
        min_padj=("padj", "min"),
        max_combined=("Combined", "max"),
        any_term=("Term", "first")
    ).reset_index()

    gp = gp.sort_values(
        ["appear", "max_combined", "min_padj"],
        ascending=[False, False, True]
    )

    return gp["any_term"].head(k).tolist()


# =========================================================

# =========================================================

MANUAL_TERMS = [
    "vascular process in circulatory system (GO:0003018)",
    "regulation of tube diameter (GO:0035296)",
    "receptor guanylyl cyclase signaling pathway (GO:0007168)",
    "negative regulation of blood pressure (GO:0045776)",
    #"leukocyte chemotaxis involved in inflammatory response (GO:0002232)",
    "positive regulation of leukocyte chemotaxis (GO:0002690)",
    "positive regulation of monocyte chemotaxis (GO:0090026)",
    "granulocyte activation (GO:0036230)",
    "neutrophil activation (GO:0042119)",
    "regulation of tissue remodeling (GO:0034103)",
]

USE_MANUAL = True

if USE_MANUAL:
    selected_terms = MANUAL_TERMS
else:
    selected_terms = select_terms_auto(dfs_full, k=10)


seen = set()
selected_terms = [
    t for t in selected_terms
    if not (term_key(t) in seen or seen.add(term_key(t)))
]

print(f"[INFO] Selected GO terms: {len(selected_terms)}")


# =========================================================
# 4) Venn gene sets
# =========================================================

CHD_genes = gene_sets["CHD"]
Stroke_genes = gene_sets["Stroke"]

CHD_color = "#C97C5D"
Stroke_color = "#6B8F71"
deep_red = "#B00020"


def mix(colors):
    return np.mean([mcolors.to_rgb(c) for c in colors], axis=0)


region_color_map = {
    "10": CHD_color,
    "01": Stroke_color,
    "11": deep_red,
}


def region_of_gene(g):
    return ("1" if g in CHD_genes else "0") + \
           ("1" if g in Stroke_genes else "0")


# =========================================================

# =========================================================

def parse_genes_field(x):
    if pd.isna(x):
        return []

    return [
        t.strip()
        for t in str(x).replace(",", ";").split(";")
        if t.strip()
    ]


gene_lists = {}
gene_counts = {}

for term in selected_terms:
    genes = set()

    for dis, df in dfs_full.items():

        tmp = df[df["Term"] == term]

        if len(tmp) == 0:
            continue

        for g in tmp["Genes"].dropna():
            genes.update(parse_genes_field(g))

    gene_lists[term] = sorted(genes)
    gene_counts[term] = len(genes)


# =========================================================



# =========================================================

bubble = []

for dis, df in dfs_full.items():
    for term in selected_terms:

        row = df[df["Term"] == term]

        if len(row) == 0:
            bubble.append((dis, term, None, 0))
        else:
            padj = float(row["Adjusted P-value"].values[0])

            gcount = 0
            if "Genes" in row.columns and pd.notna(row["Genes"].values[0]):
                gcount = len(parse_genes_field(row["Genes"].values[0]))

            bubble.append((dis, term, -np.log10(padj + 1e-300), gcount))


# color norm
vals = [
    v for (_, _, v, _) in bubble
    if v is not None and np.isfinite(v)
]

vmin = float(np.min(vals)) if len(vals) else 0.0
vmax = float(np.max(vals)) if len(vals) else 1.0

if vmin == vmax:
    vmax = vmin + 1.0

norm = Normalize(vmin=vmin, vmax=vmax)


# =========================================================

# =========================================================

fig = plt.figure(figsize=(10, 9), constrained_layout=True)


# ---------------------------------------------------------
# A. Venn
# ---------------------------------------------------------

ax1 = plt.subplot2grid((10, 1), (0, 0), rowspan=2)

v = venn2(
    [CHD_genes, Stroke_genes],
    set_labels=("Coronary Heart Disease", "Ischemic Stroke"),
    ax=ax1
)

for k, c in region_color_map.items():
    p = v.get_patch_by_id(k)
    if p:
        p.set_color(c)
        p.set_alpha(1.0)
        p.set_edgecolor("white")
        p.set_linewidth(0.8)


lbl = v.get_label_by_id("11")
if lbl:
    lbl.set_color("white")


if v.set_labels is not None:
    for txt in v.set_labels:
        if txt:
            txt.set_fontweight("bold")
            txt.set_fontsize(13)

if v.subset_labels is not None:
    for txt in v.subset_labels:
        if txt:
            txt.set_fontweight("bold")
            txt.set_fontsize(12)

ax1.set_title("Soma top10% gene overlap", fontsize=18)


# ---------------------------------------------------------
# B. Gene barplot
# ---------------------------------------------------------

ax2 = plt.subplot2grid((10, 1), (2, 0), rowspan=4)

terms = selected_terms
x = np.arange(len(terms))
counts = [gene_counts[t] for t in terms]

ax2.bar(x, counts, color="lightgray")

ax2.set_ylabel("Gene Count")
ax2.tick_params(axis="x", bottom=False, labelbottom=False)
ax2.set_xlim(-0.5, len(terms) - 0.5)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

TOP_GENES_TO_SHOW = 8

for i, term in enumerate(terms):

    genes = gene_lists[term][:TOP_GENES_TO_SHOW]

    
    genes = sorted(
        genes,
        key=lambda g: 0 if region_of_gene(g) == "11" else 1
    )

    for j, g in enumerate(genes):

        region = region_of_gene(g)
        color = region_color_map.get(region, "black")

        ax2.text(
            i,
            j * 0.9 + 0.6,
            g,
            fontsize=8,
            ha="center",
            color=color,
            fontweight="bold" if region == "11" else "normal"
        )


# ---------------------------------------------------------
# C. Bubble plot
# ---------------------------------------------------------

ax3 = plt.subplot2grid((10, 1), (7, 0), rowspan=3, sharex=ax2)

disease_order = ["Stroke", "CHD"]
y_map = {d: i for i, d in enumerate(disease_order)}

ax3.set_ylim(-0.6, len(disease_order) - 0.4)
ax3.set_yticks(range(len(disease_order)))
ax3.set_yticklabels(["Ischemic Stroke", "Coronary Heart Disease"])

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

for dis, term, logp, gcount in bubble:

    xi = terms.index(term)
    yi = y_map[dis]

    size = max(20, gcount * 35)

    if logp is None:
        ax3.scatter(
            xi,
            yi,
            s=size,
            facecolors="none",
            edgecolors="gray",
            linewidth=0.8
        )
    else:
        ax3.scatter(
            xi,
            yi,
            s=size,
            c=[logp],
            cmap="Reds",
            norm=norm,
            edgecolors="black",
            linewidth=0.3
        )

ax3.set_xticks(x)
ax3.set_xticklabels(
    [wrap_label(clean_term(t), 18) for t in terms],
    rotation=45,
    ha="right",
    fontsize=8
)

# colorbar
sm = ScalarMappable(norm=norm, cmap="Reds")
sm.set_array([])

cbar = plt.colorbar(sm, ax=ax3, pad=0.01)
cbar.set_label("-log10(padj)")


# =========================================================
# Save
# =========================================================

out_pdf = os.path.join(
    OUTDIR,
    "FINAL_SOMA_GO_VENN_BAR_BUBBLE_CHD_istroke.pdf"
)

out_svg = os.path.join(
    OUTDIR,
    "FINAL_SOMA_GO_VENN_BAR_BUBBLE_CHD_istroke.svg"
)

plt.savefig(out_pdf, dpi=300, transparent=True, bbox_inches="tight")
plt.savefig(out_svg, dpi=300, transparent=True, bbox_inches="tight")

print("[OK]", out_pdf, "saved")
print("[OK]", out_svg, "saved")